# P2 reviewer follow-up — complete Colab execution package

Run cells in order on **Google Colab with A100 80 GB (or comparable GPU)**. The notebook verifies the two historical XLSX hashes, reconstructs complete M1 dual-blocked predictions for T1–T4, executes size-matched M1 and M3 controls, recovers complete M3 dual-blocked OOF predictions, and evaluates the pinned Qwen3-8B judge in full precision and an independent Mistral-7B judge on Condition C.

This is **post-hoc**. It never overwrites archived reference predictions or selects a favourable result. All intermediate parts are saved in Google Drive for resume. Do not change the target definitions, model revision, folds, or scoring rule after seeing outcomes. Each experiment is individually labelled; a final audit ZIP is produced only after required counts and manifests pass. Separate runs of this notebook may be needed because M3 repeated training is long. No paper numbers are asserted before the outputs are examined.

In [ ]:
from pathlib import Path
import os, sys, subprocess, hashlib, json, time, gc, shutil, zipfile, platform
assert 'google.colab' in sys.modules, 'Run in Google Colab, not a local Jupyter kernel.'
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT=Path('/content/drive/MyDrive/P2')
INPUT=PROJECT/'input'; OUTPUT=PROJECT/'results'/'REVIEWER_GPU_20260928'
REPO=Path('/content/semantic-coherence-recovery')
for d in [INPUT,OUTPUT]:d.mkdir(parents=True,exist_ok=True)
# Edit these only BEFORE starting the analysis; partial outputs are guarded by run manifests.
M1_MATCH_REPEATS=5
M3_MATCH_REPEATS=5
TARGETS=['T1','T2','T3','T4']
BOOT_REPS=2000
SEED=20260908
RUN_M1=True
RUN_M3_DUAL=True
RUN_M3_SIZE_MATCHED=True
RUN_QWEN_FULL=True
RUN_MISTRAL=True
M4_CONDITIONS=['C']  # Reference Condition C on all four targets. A/B are optional new analyses.
print('Input:',INPUT,'Output:',OUTPUT)

In [ ]:
%pip -q install -U "transformers>=4.57,<5" datasets accelerate safetensors pandas openpyxl "scikit-learn>=1.4" huggingface_hub bitsandbytes

In [ ]:
import numpy as np, pandas as pd, torch, transformers
from collections import Counter
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,AutoModelForCausalLM,
    DataCollatorWithPadding,Trainer,TrainingArguments,EarlyStoppingCallback,set_seed,BitsAndBytesConfig)
from huggingface_hub import HfApi
assert torch.cuda.is_available(), 'Select GPU runtime. A100 80 GB recommended for BF16 Qwen3-8B.'
print('GPU',torch.cuda.get_device_name(0),'VRAM GiB',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
print('Torch',torch.__version__,'Transformers',transformers.__version__)
if not REPO.exists():
 subprocess.run(['git','clone','--depth','1','--branch','ipm-submission-prep-2026-09-18',
    'https://github.com/fatmas1982/semantic-coherence-recovery.git',str(REPO)],check=True)
 subprocess.run([sys.executable,str(REPO/'UNPACK_RELEASE.py')],check=True)
else:
 assert (REPO/'data/computational/m3/OOF_M3_T1_C_seed20260908.csv').exists(), 'Run UNPACK_RELEASE.py in repository.'
repo_sha=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Repository commit',repo_sha)

In [ ]:
GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED_HASH={GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
 BASE_NAME:'4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163'}
EXPECTED_N={'T1':720,'T2':474,'T3':460,'T4':474}
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1<<20),b''):h.update(block)
 return h.hexdigest()
for name,expected in EXPECTED_HASH.items():
 path=INPUT/name
 assert path.is_file(), f'Upload {name} to {INPUT} before continuing.'
 observed=sha(path);print(name,observed)
 assert observed==expected, f'WRONG INPUT VERSION: {name}; stop; do not bypass.'
gold=pd.read_excel(INPUT/GOLD_NAME,sheet_name='FINAL_GOLD').reset_index(drop=True)
assert len(gold)==720 and gold.Case_ID.is_unique and gold.Prompt_ID.nunique()==120
assert gold.groupby('Prompt_ID',sort=False).size().eq(6).all()
gold['Generator_Stratum']=gold.groupby('Prompt_ID',sort=False).cumcount().astype(int)
plan=pd.read_csv(REPO/'data/prompts/paper2_generation_plan_720.csv')
check=gold[['Case_ID','Generator_Stratum']].merge(plan[['case_id','model_id']],left_on='Case_ID',right_on='case_id',validate='one_to_one')
assert len(check)==720 and check.groupby('Generator_Stratum').model_id.nunique().eq(1).all()
assert check.loc[check.Generator_Stratum.eq(0),'model_id'].eq('M01').all()
GOLD_BY_CASE=gold.set_index('Case_ID',drop=False)
oof={t:pd.read_excel(INPUT/BASE_NAME,sheet_name='OOF_'+t) for t in TARGETS}
for t,d in oof.items():
 assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID).issubset(set(gold.Case_ID))
 assert set(d.Fold.astype(int))==set(range(5)) and set(d.y_true.astype(int))==({1,2,3} if t=='T4' else {0,1})
 merged=d[['Case_ID','Prompt_ID']].merge(gold[['Case_ID','Prompt_ID']],on='Case_ID',suffixes=('_oof','_gold'),validate='one_to_one')
 assert merged.Prompt_ID_oof.eq(merged.Prompt_ID_gold).all()
 oof[t]=d.merge(gold[['Case_ID','Generator_Stratum']],on='Case_ID',validate='one_to_one')
 ref=pd.read_csv(REPO/f'data/computational/m3/OOF_M3_{t}_C_seed20260908.csv')
 assert len(ref)==len(d) and set(ref.Case_ID)==set(d.Case_ID)
print('Validated targets:',{t:len(d) for t,d in oof.items()})
(OUTPUT/'INPUT_MANIFEST.json').write_text(json.dumps({'gold_sha256':sha(INPUT/GOLD_NAME),'base_sha256':sha(INPUT/BASE_NAME),
 'repository_commit':repo_sha,'expected_n':EXPECTED_N,'gpu':torch.cuda.get_device_name(0),
 'torch':torch.__version__,'transformers':transformers.__version__},indent=2))

## 1. Complete M1 dual block and size-matched CPU control

This stage reads **all four complete `OOF_T*` sheets** from the verified baseline workbook. It ignores the truncated published dual-block OOF file. The size-matched condition removes the same number of training rows as generator blocking, but chooses rows without using generator identity. Five fixed random repetitions run per target. The same held-out fold × generator cases are predicted in every condition.

In [ ]:
def serialise_C(g):
 prompt=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
 source=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
 output=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
 if str(g['Source_Available']).lower()=='yes' and source:
  return f'Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}'
 return f'Task:\n{prompt}\n\nResponse:\n{output}'
TEXT={str(r['Case_ID']):serialise_C(r) for _,r in gold.iterrows()}
# Exact TF-IDF/LinearSVC settings from P2_81_CPU_VALIDITY_DIAGNOSTICS.py.
def m1_fit_predict(train,test):
 labels=set(train.y_true.astype(int));assert labels==set(test['_target_classes'].iloc[0])
 vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.98,max_features=40000,sublinear_tf=True)
 xtr=vec.fit_transform([TEXT[str(x)] for x in train.Case_ID])
 xte=vec.transform([TEXT[str(x)] for x in test.Case_ID])
 clf=LinearSVC(C=1.0,class_weight='balanced',random_state=20260908)
 clf.fit(xtr,train.y_true.astype(int));return clf.predict(xte)
def selected_train(rows,fold,gen,variant,repeat,target):
 base=rows[rows.Fold.astype(int)!=fold]
 blocked=base[base.Generator_Stratum.astype(int)!=gen]
 if variant=='dual':return blocked.copy()
 assert variant=='size_matched'
 # Stable independent subset seed, invariant to Python hash randomization.
 key=f'{target}|{fold}|{gen}|{repeat}|20260928'.encode()
 seed=int(hashlib.sha256(key).hexdigest()[:8],16)
 rng=np.random.default_rng(seed)
 for attempt in range(200):
  idx=rng.choice(len(base),size=len(blocked),replace=False)
  selected=base.iloc[np.sort(idx)].copy()
  if set(selected.y_true.astype(int))==set(rows.y_true.astype(int)):
   dev_fold=(fold+1)%5
   inner=selected[selected.Fold.astype(int)!=dev_fold]
   if set(inner.y_true.astype(int))==set(rows.y_true.astype(int)):
    return selected
 raise RuntimeError('Could not obtain class-complete size-matched subset')
def run_m1():
 out=OUTPUT/'M1_COMPLETE';out.mkdir(exist_ok=True)
 config={'gold_sha256':EXPECTED_HASH[GOLD_NAME],'base_sha256':EXPECTED_HASH[BASE_NAME],
         'repeats':M1_MATCH_REPEATS,'protocol':'M1_complete_dual_and_size_matched_v1'}
 guard=out/'RUN_CONFIG.json'
 if guard.exists():assert json.loads(guard.read_text())==config, 'M1 resume configuration changed'
 else:guard.write_text(json.dumps(config,indent=2))
 all_rows=[]
 for t,rows in oof.items():
  for variant,repeats in [('dual',1),('size_matched',M1_MATCH_REPEATS)]:
   for rep in range(repeats):
    parts=[]
    for f in range(5):
     for g in range(6):
      test=rows[(rows.Fold.astype(int)==f)&(rows.Generator_Stratum==g)].copy()
      assert len(test)>0
      path=out/f'PART_{t}_{variant}_r{rep}_f{f}_g{g}.csv'
      if path.exists():
       part=pd.read_csv(path)
       assert len(part)==len(test) and set(part.Case_ID)==set(test.Case_ID)
      else:
       train=selected_train(rows,f,g,variant,rep,t)
       test['_target_classes']=[tuple(sorted(rows.y_true.astype(int).unique()))]*len(test)
       pred=m1_fit_predict(train,test)
       part=test.drop(columns=['_target_classes'])[['Case_ID','Prompt_ID','Task_Type','Fold','Generator_Stratum','y_true','Human_Easy']].copy()
       part['M1_pred']=pred;part['train_n']=len(train);part['variant']=variant;part['repeat']=rep
       tmp=path.with_suffix('.tmp');part.to_csv(tmp,index=False);os.replace(tmp,path)
      parts.append(part)
    full=pd.concat(parts,ignore_index=True)
    assert len(full)==EXPECTED_N[t] and full.Case_ID.is_unique and set(full.Case_ID)==set(rows.Case_ID)
    merged=full.merge(rows[['Case_ID','M1_pred']].rename(columns={'M1_pred':'M1_archived_pred'}),on='Case_ID',validate='one_to_one')
    labels=sorted(rows.y_true.unique())
    metric=f1_score(merged.y_true,merged.M1_pred,labels=labels,average='macro')
    base=f1_score(merged.y_true,merged.M1_archived_pred,labels=labels,average='macro')
    expected_base={'T1':.6138018360240582,'T2':.6688556657817522,'T3':.7216432461507429,'T4':.5671753725699734}
    assert abs(base-expected_base[t])<1e-8, f'Archived M1 mismatch on {t}: {base}'
    all_rows.append({'target':t,'variant':variant,'repeat':rep,'n':len(full),'macro_f1':metric,'archived_standard_m1':base})
    full.to_csv(out/f'OOF_M1_{t}_{variant}_r{rep}_C.csv',index=False)
    print(t,variant,rep,'n',len(full),'F1',round(metric,4),flush=True)
 pd.DataFrame(all_rows).to_csv(out/'M1_COMPLETE_METRICS.csv',index=False)
 (out/'MANIFEST.json').write_text(json.dumps({'input_hashes':EXPECTED_HASH,'repeat_count':M1_MATCH_REPEATS,
 'sampling_seed_scheme':'SHA256(target|fold|generator|repeat|20260928)','status':'post-hoc'},indent=2))
if RUN_M1:run_m1()

## 2. M3 ModernBERT: regenerate complete strict dual block and train-size controls

This stage uses the **pinned ModernBERT revision and original two-stage epoch selection/refit**. Each T1–T4 test fold × generator cell is predicted once per variant. The size-matched condition repeats five independent, class-valid row samples, reselects epochs on its training subset, and never uses test labels to choose the sample or epochs. This may take many Colab sessions; each completed part is saved and validated on resume. Do not lower the repeat count after inspecting partial results.

In [ ]:
SEED=20260908
M3_MODEL_ID='answerdotai/ModernBERT-base'
M3_MODEL_REVISION='c0e44438c79d5a72972fe8b14dbbc822418356c9'
MAX_LENGTH=1024
LEARNING_RATE=2e-5
WEIGHT_DECAY=.01
MAX_SELECT_EPOCHS=8
TRAIN_BATCH=8
EVAL_BATCH=16
GRAD_ACCUM=2
EARLY_STOPPING_PATIENCE=2
BOOT_REPS=2000
TARGETS_TO_RUN=TARGETS

def serialise_C(g):
    prompt=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
    source=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
    output=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
    if str(g['Source_Available']).lower()=='yes' and source:
        return f'Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}'
    return f'Task:\n{prompt}\n\nResponse:\n{output}'

tokenizer=AutoTokenizer.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION)

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self,*args,class_weights=None,**kwargs):
        super().__init__(*args,**kwargs); self.class_weights=class_weights
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop('labels'); out=model(**inputs)
        loss=torch.nn.functional.cross_entropy(out.logits,labels,weight=self.class_weights.to(out.logits.device))
        return (loss,out) if return_outputs else loss

def compute_metrics_eval(eval_pred):
    logits,labels=eval_pred
    return {'macro_f1':f1_score(labels,np.argmax(logits,axis=-1),average='macro')}

def make_dataset(rows,label_to_idx):
    return Dataset.from_dict({
        'text':[serialise_C(gold_by_case.loc[str(cid)]) for cid in rows.Case_ID],
        'labels':[label_to_idx[int(v)] for v in rows.y_true],
        'case_id':[str(x) for x in rows.Case_ID],
    })

def tokenise(ds):
    ids=list(ds['case_id'])
    def fn(batch): return tokenizer(batch['text'],truncation=True,max_length=MAX_LENGTH)
    return ds.map(fn,batched=True,remove_columns=['text','case_id']),ids

def weights_for(rows,label_to_idx):
    c=Counter(label_to_idx[int(v)] for v in rows.y_true); n=sum(c.values()); k=len(label_to_idx)
    assert set(c)==set(range(k)),f'Missing training class: {c}'
    return torch.tensor([n/(k*c[i]) for i in range(k)],dtype=torch.float32)

def selected_epoch(trainer):
    cand=[(float(x['eval_macro_f1']),float(x['epoch'])) for x in trainer.state.log_history if 'eval_macro_f1' in x and 'epoch' in x]
    assert cand
    best=max(v for v,_ in cand); epochs=[e for v,e in cand if abs(v-best)<1e-12]
    return max(1,int(round(min(epochs)))),best

In [ ]:
LOCAL_TMP=Path('/content/p2_gpu_tmp');LOCAL_TMP.mkdir(exist_ok=True)
LOCAL_CACHE=Path('/content/p2_hf_cache');LOCAL_CACHE.mkdir(exist_ok=True)
os.environ['HF_HOME']=str(LOCAL_CACHE)
# The imported original training helper uses gold_by_case.
gold_by_case=GOLD_BY_CASE
def train_cell(target,outer_fold,heldout_generator,variant,repeat):
    rows=oof[target].copy()
    labels=sorted(rows.y_true.astype(int).unique())
    label_to_idx={v:i for i,v in enumerate(labels)}; idx_to_label={i:v for v,i in label_to_idx.items()}
    test=rows[(rows.Fold.astype(int)==outer_fold)&(rows.Generator_Stratum.astype(int)==heldout_generator)].copy()
    outer_train=selected_train(rows,outer_fold,heldout_generator,variant,repeat,target)
    outer_train=outer_train.sort_values('Case_ID').copy()
    assert len(test)>0
    assert set(outer_train.y_true.astype(int).unique())==set(labels)

    available=sorted(outer_train.Fold.astype(int).unique())
    preferred=(outer_fold+1)%5
    dev_fold=preferred if preferred in available else available[0]
    inner_dev=outer_train[outer_train.Fold.astype(int)==dev_fold].copy()
    inner_train=outer_train[outer_train.Fold.astype(int)!=dev_fold].copy()
    assert set(inner_train.y_true.astype(int).unique())==set(labels)

    # Stage 1: epoch selection inside the doubly restricted population.
    tr,_=tokenise(make_dataset(inner_train,label_to_idx)); dv,_=tokenise(make_dataset(inner_dev,label_to_idx))
    select_dir=LOCAL_TMP/f'select_{target}_{variant}_r{repeat}_f{outer_fold}_g{heldout_generator}'
    shutil.rmtree(select_dir,ignore_errors=True); select_dir.mkdir(parents=True,exist_ok=True)
    set_seed(SEED)
    m=AutoModelForSequenceClassification.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION,num_labels=len(labels))
    bf=torch.cuda.is_bf16_supported()
    args=TrainingArguments(
        output_dir=str(select_dir),learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,
        num_train_epochs=MAX_SELECT_EPOCHS,per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,
        metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=1,
        bf16=bf,fp16=not bf,report_to='none',seed=SEED,data_seed=SEED,logging_steps=20,
    )
    trainer=WeightedTrainer(model=m,args=args,train_dataset=tr,eval_dataset=dv,
        data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=compute_metrics_eval,
        class_weights=weights_for(inner_train,label_to_idx),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)])
    trainer.train(); epochs,best_dev=selected_epoch(trainer)
    del trainer,m,tr,dv; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(select_dir,ignore_errors=True)

    # Stage 2: fresh refit on all restricted outer-training rows, then one test pass.
    tr,_=tokenise(make_dataset(outer_train,label_to_idx)); te,_=tokenise(make_dataset(test,label_to_idx))
    refit_dir=LOCAL_TMP/f'refit_{target}_{variant}_r{repeat}_f{outer_fold}_g{heldout_generator}'
    shutil.rmtree(refit_dir,ignore_errors=True); refit_dir.mkdir(parents=True,exist_ok=True)
    set_seed(SEED)
    m=AutoModelForSequenceClassification.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION,num_labels=len(labels))
    args=TrainingArguments(
        output_dir=str(refit_dir),learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,
        num_train_epochs=epochs,per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy='no',save_strategy='no',bf16=bf,fp16=not bf,report_to='none',
        seed=SEED,data_seed=SEED,logging_steps=20,
    )
    trainer=WeightedTrainer(model=m,args=args,train_dataset=tr,
        data_collator=DataCollatorWithPadding(tokenizer),class_weights=weights_for(outer_train,label_to_idx))
    trainer.train(); pr=trainer.predict(te)
    probs=torch.softmax(torch.tensor(pr.predictions),dim=-1).numpy(); ii=np.argmax(probs,axis=1)
    out=test[['Case_ID','Prompt_ID','Task_Type','Fold','Generator_Stratum','y_true','Human_Easy']].copy()
    out['M3_dual_pred']=[idx_to_label[int(i)] for i in ii]
    out['M3_dual_probs']=[json.dumps([float(x) for x in p]) for p in probs]
    out['train_n']=len(outer_train); out['variant']=variant; out['repeat']=repeat; out['selected_epochs']=epochs; out['inner_dev_fold']=dev_fold; out['best_inner_dev_macro_f1']=best_dev
    del trainer,m,tr,te; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(refit_dir,ignore_errors=True)
    return out

In [ ]:
def run_m3_variant(target,variant,repeat):
 out=OUTPUT/'M3_COMPLETE';out.mkdir(exist_ok=True)
 config={'gold_sha256':EXPECTED_HASH[GOLD_NAME],'base_sha256':EXPECTED_HASH[BASE_NAME],
         'revision':M3_MODEL_REVISION,'repeats':M3_MATCH_REPEATS,
         'protocol':'M3_nested_selection_dual_and_size_matched_v1'}
 guard=out/'RUN_CONFIG.json'
 if guard.exists():assert json.loads(guard.read_text())==config, 'M3 resume configuration changed'
 else:guard.write_text(json.dumps(config,indent=2))
 parts=[]
 for f in range(5):
  for g in range(6):
   path=out/f'PART_{target}_{variant}_r{repeat}_f{f}_g{g}.csv'
   expected=set(oof[target].loc[(oof[target].Fold.astype(int)==f)&(oof[target].Generator_Stratum==g),'Case_ID'])
   if path.exists():
    part=pd.read_csv(path)
    assert len(part)==len(expected) and set(part.Case_ID)==expected
    assert set(part.variant)=={variant} and set(part.repeat)=={repeat}
   else:
    print('M3 train',target,variant,repeat,f,g,flush=True)
    part=train_cell(target,f,g,variant,repeat)
    assert set(part.Case_ID)==expected
    tmp=path.with_suffix('.tmp');part.to_csv(tmp,index=False);os.replace(tmp,path)
   parts.append(part)
 full=pd.concat(parts,ignore_index=True)
 assert len(full)==EXPECTED_N[target] and full.Case_ID.is_unique and set(full.Case_ID)==set(oof[target].Case_ID)
 full.to_csv(out/f'OOF_M3_{target}_{variant}_r{repeat}_C.csv',index=False)
 return full
m3_runs=[]
if RUN_M3_DUAL or RUN_M3_SIZE_MATCHED:
 for t in TARGETS:
  variants=[]
  if RUN_M3_DUAL:variants.append(('dual',1))
  if RUN_M3_SIZE_MATCHED:variants.append(('size_matched',M3_MATCH_REPEATS))
  for variant,repeats in variants:
   for rep in range(repeats):
    full=run_m3_variant(t,variant,rep)
    labels=sorted(oof[t].y_true.unique())
    ref=pd.read_csv(REPO/f'data/computational/m3/OOF_M3_{t}_C_seed20260908.csv')[['Case_ID','M3_pred']]
    full=full.merge(ref.rename(columns={'M3_pred':'M3_standard_pred'}),on='Case_ID',validate='one_to_one')
    m3_runs.append({'target':t,'variant':variant,'repeat':rep,'n':len(full),
      'macro_f1':f1_score(full.y_true,full.M3_dual_pred,labels=labels,average='macro'),
      'standard_macro_f1':f1_score(full.y_true,full.M3_standard_pred,labels=labels,average='macro')})
    pd.DataFrame(m3_runs).to_csv(OUTPUT/'M3_COMPLETE'/'M3_RUN_METRICS.csv',index=False)
    print('COMPLETE',m3_runs[-1],flush=True)
 (OUTPUT/'M3_COMPLETE'/'MANIFEST.json').write_text(json.dumps({'status':'post-hoc','model':M3_MODEL_ID,
  'revision':M3_MODEL_REVISION,'gold_sha256':EXPECTED_HASH[GOLD_NAME],
  'base_sha256':EXPECTED_HASH[BASE_NAME],'size_matched_repeats':M3_MATCH_REPEATS,
  'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'transformers':transformers.__version__},indent=2))

## 3. M4 judges: pinned Qwen3-8B full precision and different-family Mistral-7B

Both new runs use the **original P1/identity target rubric, same gold-defined eligible cases, chat template and next-token A/B/C forced-choice rule**, under Condition C. Qwen is loaded at its historical pinned revision in BF16 (or FP16 if BF16 unsupported); Mistral's exact hub revision is resolved and recorded *before* loading. A single-token code check is mandatory: if a model tokenizer fails it, the run stops rather than silently changing the decision rule. These are new post-hoc comparisons, not a rerun of the original 4-bit NF4 score. One target CSV is saved after each batch, so interrupted runs resume.

In [ ]:
def serialise(g,condition):
    prompt=str(g["Prompt_Text"] if pd.notna(g["Prompt_Text"]) else "").strip()
    source=str(g["Source_Context"] if pd.notna(g["Source_Context"]) else "").strip()
    output=str(g["Model_Output"] if pd.notna(g["Model_Output"]) else "").strip()
    if condition=="A":
        return f"Response:\n{output}"
    if condition=="B":
        return f"Task:\n{prompt}\n\nResponse:\n{output}"
    if condition=="C":
        if str(g["Source_Available"]).lower()=="yes" and source:
            return f"Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}"
        return f"Task:\n{prompt}\n\nResponse:\n{output}"
    raise ValueError(condition)

TARGETS = {
"T1":{
    "codes":["A","B"],
    "map":{"A":0,"B":1},
    "rubric":"""Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.
A = no_failure: no identifiable failure under the supplied evidence.
B = failure: at least one identifiable semantic-coherence or authorised-grounding failure.
Do not classify the subtype. Choose only A or B."""
},
"T2":{
    "codes":["A","B"],
    "map":{"A":0,"B":1},
    "rubric":"""The case is already known to contain a failure. Decide whether authorised grounding is involved.
A = semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.
B = grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.
Choose only A or B."""
},
"T3":{
    "codes":["A","B"],
    "map":{"A":0,"B":1},
    "rubric":"""The case belongs to one of two mechanism families.
A = Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.
B = OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.
Choose only A or B."""
},
"T4":{
    "codes":["A","B","C"],
    "map":{"A":1,"B":2,"C":3},
    "rubric":"""The case is already known to contain a failure. Judge functional severity, not confidence.
A = severity 1 (Minor): localized disruption with limited effect on the main answer.
B = severity 2 (Moderate): materially weakens an important task-relevant relation or component.
C = severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.
Choose only A, B, or C."""
},
}

SYSTEM = """You are a conservative semantic-coherence evaluator.
Apply only the supplied target rubric to the visible text.
Do not infer hidden facts. Do not output an explanation.
Your decision space is closed: select exactly one of the permitted answer codes."""

In [ ]:
M4_MODEL_CONFIGS=[
 {'name':'qwen_full','id':'Qwen/Qwen3-8B','revision':'e8bbd8252970581ea5b08b6a5b3e668adaf3161a','run':RUN_QWEN_FULL},
 {'name':'mistral_full','id':'mistralai/Mistral-7B-Instruct-v0.3','revision':None,'run':RUN_MISTRAL},
]
# At least 35 GB free VRAM is required by this notebook before loading either full-precision judge.
assert torch.cuda.get_device_properties(0).total_memory/2**30>=35, 'Use an A100 40/80 GB or H100 for full-precision runs.'
def load_judge(cfg):
 global tokenizer,model,CODE_TOKEN_ID
 revision=cfg['revision'] or HfApi().model_info(cfg['id']).sha
 assert len(revision)==40, f'No immutable model revision resolved for {cfg["id"]}'
 cfg['resolved_revision']=revision
 dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
 tokenizer=AutoTokenizer.from_pretrained(cfg['id'],revision=revision)
 tokenizer.padding_side='left'
 if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
 CODE_TOKEN_ID={}
 for c in ['A','B','C']:
  ids=tokenizer(c,add_special_tokens=False)['input_ids']
  assert len(ids)==1, f'{cfg["name"]}: code {c} is not one token ({ids}); stop and report.'
  CODE_TOKEN_ID[c]=ids[0]
 model=AutoModelForCausalLM.from_pretrained(cfg['id'],revision=revision,torch_dtype=dtype,device_map='auto')
 assert not hasattr(model,'hf_device_map') or all('cpu' not in str(v) and 'disk' not in str(v) for v in model.hf_device_map.values()),'CPU/disk offload detected'
 model.eval()
 print('Loaded',cfg['id'],revision,'dtype',dtype)
 return revision,str(dtype)
def judge_chat(row,target,condition,model_name):
 user=TARGETS[target]['rubric']+'\n\nVisible case:\n'+serialise(row,condition)
 if model_name=='qwen_full':
  msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':user}]
 else:
  # Mistral template compatibility: preserve the instruction text in one user turn.
  msgs=[{'role':'user','content':SYSTEM+'\n\n'+user}]
 kwargs={'tokenize':False,'add_generation_prompt':True}
 if model_name=='qwen_full':kwargs['enable_thinking']=False
 return tokenizer.apply_chat_template(msgs,**kwargs)
@torch.inference_mode()
def score_batch(texts,codes):
 enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=4096)
 enc={k:v.to(model.device) for k,v in enc.items()}
 logits=model(**enc).logits[:,-1,:].float()
 logp=torch.log_softmax(logits,dim=-1)
 ids=torch.tensor([CODE_TOKEN_ID[c] for c in codes],device=logp.device)
 return logp[:,ids].detach().cpu().numpy()
def run_judge_target(cfg,target,condition,batch_size=4):
 outdir=OUTPUT/'M4_NEW'/cfg['name'];outdir.mkdir(parents=True,exist_ok=True)
 path=outdir/f'OOF_{cfg["name"]}_{target}_{condition}.csv'
 rows=oof[target].copy();expected=set(rows.Case_ID.astype(str))
 old=pd.read_csv(path) if path.exists() else pd.DataFrame()
 if len(old):
  assert old.Case_ID.is_unique and set(old.Case_ID).issubset(expected)
  assert set(old.model_revision)=={cfg['resolved_revision']}
 existing=set(old.Case_ID.astype(str)) if len(old) else set()
 records=old.to_dict('records') if len(old) else []
 todo=rows[~rows.Case_ID.astype(str).isin(existing)]
 codes=TARGETS[target]['codes']
 for start in range(0,len(todo),batch_size):
  subset=todo.iloc[start:start+batch_size]
  texts=[judge_chat(GOLD_BY_CASE.loc[str(r.Case_ID)],target,condition,cfg['name']) for r in subset.itertuples()]
  score=score_batch(texts,codes)
  for r,sc in zip(subset.itertuples(),score):
   i=int(np.argmax(sc));cc=codes[i]
   records.append({'Case_ID':str(r.Case_ID),'Prompt_ID':r.Prompt_ID,'Task_Type':r.Task_Type,
      'Fold':int(r.Fold),'y_true':int(r.y_true),'Generator_Stratum':int(r.Generator_Stratum),
      'pred':int(TARGETS[target]['map'][cc]),'code':cc,
      'scores':json.dumps({c:float(v) for c,v in zip(codes,sc)}),
      'model_revision':cfg['resolved_revision'],'model_id':cfg['id'],'condition':condition})
  tmp=path.with_suffix('.tmp');pd.DataFrame(records).to_csv(tmp,index=False);os.replace(tmp,path)
 full=pd.read_csv(path)
 assert len(full)==EXPECTED_N[target] and full.Case_ID.is_unique and set(full.Case_ID)==expected
 return full
m4_rows=[]
for cfg in M4_MODEL_CONFIGS:
 if not cfg['run']:continue
 revision,dtype=load_judge(cfg)
 outdir=OUTPUT/'M4_NEW'/cfg['name'];outdir.mkdir(parents=True,exist_ok=True)
 guard=outdir/'RUN_CONFIG.json'
 config={'model_id':cfg['id'],'revision':revision,'gold_sha256':EXPECTED_HASH[GOLD_NAME],
         'base_sha256':EXPECTED_HASH[BASE_NAME],'conditions':M4_CONDITIONS,'quantization':None}
 if guard.exists():assert json.loads(guard.read_text())==config, 'M4 resume configuration changed'
 else:guard.write_text(json.dumps(config,indent=2))
 (outdir/'MANIFEST.json').write_text(json.dumps({'model_id':cfg['id'],'revision':revision,
  'dtype':dtype,'quantization':None,'condition_set':M4_CONDITIONS,'rubric':'archived P1/identity',
  'scoring':'single-token next-token log probability','chat_roles':'system+user for Qwen; combined user for Mistral','gold_sha256':EXPECTED_HASH[GOLD_NAME],
  'base_sha256':EXPECTED_HASH[BASE_NAME],'gpu':torch.cuda.get_device_name(0),
  'torch':torch.__version__,'transformers':transformers.__version__},indent=2))
 for cond in M4_CONDITIONS:
  for t in TARGETS:
   full=run_judge_target(cfg,t,cond)
   labels=sorted(oof[t].y_true.unique())
   row={'model':cfg['name'],'target':t,'condition':cond,'n':len(full),
        'macro_f1':f1_score(full.y_true,full.pred,labels=labels,average='macro'),
        'accuracy':accuracy_score(full.y_true,full.pred)}
   if t=='T4':row['qwk']=cohen_kappa_score(full.y_true,full.pred,weights='quadratic')
   m4_rows.append(row)
   pd.DataFrame(m4_rows).to_csv(OUTPUT/'M4_NEW'/'M4_NEW_METRICS.csv',index=False)
   print(row,flush=True)
 del model,tokenizer;gc.collect();torch.cuda.empty_cache()

## 4. Cross-run checks, cluster uncertainty, and return ZIP

Run this cell only when all chosen stages have completed. It verifies **720/474/460/474 unique Case_IDs** for each target and each run, computes task-stratified Prompt_ID bootstrap intervals for the new judge scores, archives all OOF CSVs and manifests, and downloads one ZIP. If any stage is missing, it stops with an explicit error rather than returning an apparently complete package.

In [ ]:
def macro_cm(cm):
 tp=np.diag(cm);den=cm.sum(0)+cm.sum(1)
 return float(np.divide(2*tp,den,out=np.zeros(len(tp),float),where=den!=0).mean())
def cluster_ci(df,pred_col,labels,reps=BOOT_REPS,seed=20260928):
 idx={v:i for i,v in enumerate(labels)}
 ids=df.Prompt_ID.drop_duplicates().tolist();pos={p:i for i,p in enumerate(ids)}
 cm=np.zeros((len(ids),len(labels),len(labels)),int)
 for row in df.itertuples():cm[pos[row.Prompt_ID],idx[int(row.y_true)],idx[int(getattr(row,pred_col))]]+=1
 strata=[np.array([pos[p] for p in q.Prompt_ID.drop_duplicates()]) for _,q in df.groupby('Task_Type',sort=True)]
 rng=np.random.default_rng(seed)
 vals=[macro_cm(cm[np.concatenate([rng.choice(a,size=len(a),replace=True) for a in strata])].sum(0)) for _ in range(reps)]
 return tuple(np.percentile(vals,[2.5,97.5]))
checks=[]
if RUN_M1:
 for t in TARGETS:
  for v,reps in [('dual',1),('size_matched',M1_MATCH_REPEATS)]:
   for r in range(reps):
    f=OUTPUT/'M1_COMPLETE'/f'OOF_M1_{t}_{v}_r{r}_C.csv';assert f.exists(),f'Missing {f}'
    d=pd.read_csv(f);assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID)==set(oof[t].Case_ID)
    lo,hi=cluster_ci(d,'M1_pred',sorted(oof[t].y_true.unique()))
    checks.append({'stage':'M1','target':t,'variant':v,'repeat':r,'n':len(d),'ci_low':lo,'ci_high':hi})
if RUN_M3_DUAL or RUN_M3_SIZE_MATCHED:
 for t in TARGETS:
  for v,reps in [('dual',1 if RUN_M3_DUAL else 0),('size_matched',M3_MATCH_REPEATS if RUN_M3_SIZE_MATCHED else 0)]:
   for r in range(reps):
    f=OUTPUT/'M3_COMPLETE'/f'OOF_M3_{t}_{v}_r{r}_C.csv';assert f.exists(),f'Missing {f}'
    d=pd.read_csv(f);assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID)==set(oof[t].Case_ID)
    lo,hi=cluster_ci(d,'M3_dual_pred',sorted(oof[t].y_true.unique()))
    checks.append({'stage':'M3','target':t,'variant':v,'repeat':r,'n':len(d),'ci_low':lo,'ci_high':hi})
for cfg in M4_MODEL_CONFIGS:
 if not cfg['run']:continue
 for c in M4_CONDITIONS:
  for t in TARGETS:
   f=OUTPUT/'M4_NEW'/cfg['name']/f'OOF_{cfg["name"]}_{t}_{c}.csv';assert f.exists(),f'Missing {f}'
   d=pd.read_csv(f);assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID)==set(oof[t].Case_ID)
   lo,hi=cluster_ci(d,'pred',sorted(oof[t].y_true.unique()))
   checks.append({'stage':'M4','target':t,'variant':cfg['name']+'_'+c,'repeat':0,'n':len(d),'ci_low':lo,'ci_high':hi})
summary=pd.DataFrame(checks)
summary.to_csv(OUTPUT/'FINAL_COMPLETENESS_AND_CI.csv',index=False)
# Paired task-stratified Prompt_ID bootstrap deltas, retaining the same case population.
def paired_ci(frame,pred_a,pred_b,labels,reps=BOOT_REPS,seed=20260928):
 ix={v:j for j,v in enumerate(labels)}
 ids=frame.Prompt_ID.drop_duplicates().tolist();loc={pid:j for j,pid in enumerate(ids)}
 a=np.zeros((len(ids),len(labels),len(labels)),int);b=np.zeros_like(a)
 for row in frame.itertuples():
  k=loc[row.Prompt_ID];a[k,ix[int(row.y_true)],ix[int(getattr(row,pred_a))]]+=1
  b[k,ix[int(row.y_true)],ix[int(getattr(row,pred_b))]]+=1
 strata=[np.array([loc[x] for x in z.Prompt_ID.drop_duplicates()]) for _,z in frame.groupby('Task_Type',sort=True)]
 rng=np.random.default_rng(seed)
 vals=[]
 for _ in range(reps):
  sel=np.concatenate([rng.choice(z,size=len(z),replace=True) for z in strata])
  vals.append(macro_cm(b[sel].sum(0))-macro_cm(a[sel].sum(0)))
 point=macro_cm(b.sum(0))-macro_cm(a.sum(0))
 return point,*np.percentile(vals,[2.5,97.5])
contrasts=[]
for t in TARGETS:
 labels=sorted(oof[t].y_true.unique())
 if RUN_M1:
  baseline=oof[t][['Case_ID','Prompt_ID','Task_Type','y_true','M1_pred']].rename(columns={'M1_pred':'std'})
  candidates=[('dual',0)]+[('size_matched',r) for r in range(M1_MATCH_REPEATS)]
  for v,r in candidates:
   trial=pd.read_csv(OUTPUT/'M1_COMPLETE'/f'OOF_M1_{t}_{v}_r{r}_C.csv')[['Case_ID','M1_pred']]
   pair=baseline.merge(trial,on='Case_ID',validate='one_to_one').rename(columns={'M1_pred':'trial'})
   point,lo,hi=paired_ci(pair,'std','trial',labels)
   contrasts.append({'stage':'M1','target':t,'contrast':'standard_to_'+v,'repeat':r,'n':len(pair),'delta':point,'ci_low':lo,'ci_high':hi})
 if RUN_M3_DUAL or RUN_M3_SIZE_MATCHED:
  baseline=pd.read_csv(REPO/f'data/computational/m3/OOF_M3_{t}_C_seed20260908.csv')
  baseline=baseline[['Case_ID','Prompt_ID','Task_Type','y_true','M3_pred']].rename(columns={'M3_pred':'std'})
  candidates=([('dual',0)] if RUN_M3_DUAL else [])+([('size_matched',r) for r in range(M3_MATCH_REPEATS)] if RUN_M3_SIZE_MATCHED else [])
  for v,r in candidates:
   trial=pd.read_csv(OUTPUT/'M3_COMPLETE'/f'OOF_M3_{t}_{v}_r{r}_C.csv')[['Case_ID','M3_dual_pred']]
   pair=baseline.merge(trial,on='Case_ID',validate='one_to_one').rename(columns={'M3_dual_pred':'trial'})
   point,lo,hi=paired_ci(pair,'std','trial',labels)
   contrasts.append({'stage':'M3','target':t,'contrast':'standard_to_'+v,'repeat':r,'n':len(pair),'delta':point,'ci_low':lo,'ci_high':hi})
  if RUN_M3_DUAL and RUN_M3_SIZE_MATCHED:
   dual=pd.read_csv(OUTPUT/'M3_COMPLETE'/f'OOF_M3_{t}_dual_r0_C.csv')[['Case_ID','Prompt_ID','Task_Type','y_true','M3_dual_pred']].rename(columns={'M3_dual_pred':'dual'})
   for r in range(M3_MATCH_REPEATS):
    trial=pd.read_csv(OUTPUT/'M3_COMPLETE'/f'OOF_M3_{t}_size_matched_r{r}_C.csv')[['Case_ID','M3_dual_pred']]
    pair=dual.merge(trial,on='Case_ID',validate='one_to_one').rename(columns={'M3_dual_pred':'matched'})
    point,lo,hi=paired_ci(pair,'matched','dual',labels)
    contrasts.append({'stage':'M3','target':t,'contrast':'size_matched_to_dual','repeat':r,'n':len(pair),'delta':point,'ci_low':lo,'ci_high':hi})
 for cfg in M4_MODEL_CONFIGS:
  if not cfg['run']:continue
  for cond in M4_CONDITIONS:
   archived_path=REPO/f'data/computational/m4/OOF_M4_FORCED_{t}_{cond}.csv'
   if not archived_path.exists():continue
   old=pd.read_csv(archived_path)
   new=pd.read_csv(OUTPUT/'M4_NEW'/cfg['name']/f'OOF_{cfg["name"]}_{t}_{cond}.csv')
   pair=old[['Case_ID','Prompt_ID','Task_Type','y_true','M4_pred']].merge(new[['Case_ID','pred','Generator_Stratum']],on='Case_ID',validate='one_to_one').rename(columns={'M4_pred':'std','pred':'trial'})
   assert len(pair)==EXPECTED_N[t]
   for group,part in [('all',pair),('exclude_M01',pair[pair.Generator_Stratum!=0]),('M01',pair[pair.Generator_Stratum==0])]:
    point,lo,hi=paired_ci(part,'std','trial',labels)
    contrasts.append({'stage':'M4','target':t,'contrast':'archived_NF4_to_'+cfg['name']+'_'+cond+'_'+group,'repeat':0,'n':len(part),'delta':point,'ci_low':lo,'ci_high':hi})
pd.DataFrame(contrasts).to_csv(OUTPUT/'PAIRED_CONTRASTS.csv',index=False)
repeat_rows=[]
for stage in ['M1','M3']:
 for t in TARGETS:
  vals=[x['macro_f1'] for x in (pd.read_csv(OUTPUT/('M1_COMPLETE' if stage=='M1' else 'M3_COMPLETE')/('M1_COMPLETE_METRICS.csv' if stage=='M1' else 'M3_RUN_METRICS.csv')).to_dict('records')) if x['target']==t and x['variant']=='size_matched'] if (RUN_M1 if stage=='M1' else RUN_M3_SIZE_MATCHED) else []
  if vals:repeat_rows.append({'stage':stage,'target':t,'repeats':len(vals),'mean_macro_f1':float(np.mean(vals)),'sd_macro_f1':float(np.std(vals,ddof=1))})
pd.DataFrame(repeat_rows).to_csv(OUTPUT/'REPEAT_SUMMARY.csv',index=False)
zip_path=PROJECT/'P2_REVIEWER_GPU_COMPLETE_RESULTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in sorted(OUTPUT.rglob('*')):
  if f.is_file():z.write(f,arcname=f.relative_to(OUTPUT))
print('COMPLETE:',zip_path,'bytes',zip_path.stat().st_size)
from google.colab import files
files.download(str(zip_path))